<a href="https://colab.research.google.com/github/Rishithasree05/cyber-bullying-detection/blob/main/Bullying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — install & imports
!pip install -q imbalanced-learn

import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
# Cell 2 — load file (change filenames if needed)
# Upload the csv to /content in Colab via left Files pane or use kaggle API
df = pd.read_csv('/content/cyberbullying_tweets.csv')
df.head()


,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying


In [ ]:
# Cell 3 — inspect labels and rename columns if necessary
print("Columns:", df.columns)
# Example normalisation: assume text in 'tweet' or 'text' and label in 'class' or 'label'
# Renaming the actual columns in the dataframe
df.rename(columns={'tweet_text':'text', 'cyberbullying_type':'label'}, inplace=True)

print(df['label'].value_counts())


Columns: Index(['tweet_text', 'cyberbullying_type'], dtype='object')
label
religion               7998
age                    7992
gender                 7973
ethnicity              7961
not_cyberbullying      7945
other_cyberbullying    7823
Name: count, dtype: int64


In [ ]:
# Cell 4 — preprocessing
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(s):
    s = str(s)
    s = s.lower()
    s = re.sub(r'http\S+|www\S+',' ', s)        # remove urls
    s = re.sub(r'@\w+',' ', s)                  # remove mentions
    s = re.sub(r'[^a-z\s]',' ', s)              # keep letters only
    tokens = nltk.word_tokenize(s)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t)>1]
    return " ".join(tokens)

df['clean'] = df['text'].apply(clean_text)
df[['text','clean','label']].head()


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,text,clean,label
0,"In other words #katandandre, your food was cra...",word katandandre food crapilicious mkr,not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,aussietv white mkr theblock imacelebrityau tod...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,classy whore red velvet cupcake,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",meh thanks head concerned another angry dude t...,not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,isi account pretending kurdish account like is...,not_cyberbullying


In [ ]:
# Cell 5 — label encoding
le = LabelEncoder()
df['label'] = df['label'].astype(str)  # ensure string
print("Unique labels:", df['label'].unique())
df['label_encoded'] = le.fit_transform(df['label'])
print(dict(zip(le.classes_, le.transform(le.classes_))))

# Balance dataset (oversample minority classes)
X = df['clean']
y = df['label_encoded']

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X.to_frame(), y)
X_res = X_res['clean']
print("Balanced counts:", np.bincount(y_res))


Unique labels: ['not_cyberbullying' 'gender' 'religion' 'other_cyberbullying' 'age'
 'ethnicity']
{'age': np.int64(0), 'ethnicity': np.int64(1), 'gender': np.int64(2), 'not_cyberbullying': np.int64(3), 'other_cyberbullying': np.int64(4), 'religion': np.int64(5)}
Balanced counts: [7998 7998 7998 7998 7998 7998]


In [ ]:
# Cell 6 — vectorize & split
vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1,2))
X_vec = vectorizer.fit_transform(X_res)

X_train, X_test, y_train, y_test = train_test_split(X_vec, y_res, test_size=0.2, random_state=42, stratify=y_res)
print("Train/Test shapes:", X_train.shape, X_test.shape)


Train/Test shapes: (38390, 15000) (9598, 15000)


In [ ]:
# Cell 7 — train a multiclass classifier
clf = LogisticRegression(max_iter=1000, multi_class='auto', solver='saga', C=1.0)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("Classification report:\n", classification_report(y_test, y_pred, target_names=le.classes_))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Classification report:
                      precision    recall  f1-score   support

                age       0.95      0.97      0.96      1599
          ethnicity       0.97      0.97      0.97      1599
             gender       0.92      0.83      0.88      1600
  not_cyberbullying       0.60      0.52      0.56      1600
other_cyberbullying       0.58      0.71      0.64      1600
           religion       0.96      0.95      0.96      1600

           accuracy                           0.83      9598
          macro avg       0.83      0.83      0.83      9598
       weighted avg       0.83      0.83      0.83      9598



In [ ]:
# Cell 8 — save artifacts
joblib.dump(clf, '/content/cyber_model.pkl')
joblib.dump(vectorizer, '/content/tfidf_vectorizer.pkl')
joblib.dump(le, '/content/label_encoder.pkl')
print("Saved model and vectorizer.")


Saved model and vectorizer.


In [ ]:
# Cell 9 — severity helper
def severity_from_prob(prob):
    # prob: highest class probability
    if prob >= 0.85:
        return "HIGH"
    elif prob >= 0.6:
        return "MEDIUM"
    else:
        return "LOW"


In [ ]:
# Cell 10 — real-time demo
clf = joblib.load('/content/cyber_model.pkl')
vectorizer = joblib.load('/content/tfidf_vectorizer.pkl')
le = joblib.load('/content/label_encoder.pkl')

print("Real-time Cyberbullying Detector: type 'exit' to stop.")
while True:
    text = input("\nEnter text: ")
    if text.strip().lower() == 'exit':
        break
    clean = clean_text(text)
    vec = vectorizer.transform([clean])
    probs = clf.predict_proba(vec)[0]
    idx = np.argmax(probs)
    label = le.inverse_transform([idx])[0]
    prob = probs[idx]
    severity = severity_from_prob(prob)

    print(f"\nPredicted Category: {label}")
    print(f"Confidence: {prob:.2f}  -> Severity: {severity}")
    # Recommended action text
    if severity == "HIGH":
        print("Suggested Action: Immediate review & block/report. Notify admin/parents.")
    elif severity == "MEDIUM":
        print("Suggested Action: Flag for review and monitor closely.")
    else:
        print("Suggested Action: Log and monitor.")


Real-time Cyberbullying Detector: type 'exit' to stop.

Enter text: She should stay quiet; women don’t understand this stuff.

Predicted Category: other_cyberbullying
Confidence: 0.36  -> Severity: LOW
Suggested Action: Log and monitor.

Enter text: People of that religion are disgusting and don’t belong here.

Predicted Category: religion
Confidence: 0.41  -> Severity: LOW
Suggested Action: Log and monitor.

Enter text: I’m going to find you and make you regret ever posting that.

Predicted Category: other_cyberbullying
Confidence: 0.39  -> Severity: LOW
Suggested Action: Log and monitor.

Enter text: You’re so old, nobody wants someone your age on the team

Predicted Category: other_cyberbullying
Confidence: 0.38  -> Severity: LOW
Suggested Action: Log and monitor.

Enter text: That race is lazy and should be sent back.

Predicted Category: not_cyberbullying
Confidence: 0.69  -> Severity: MEDIUM
Suggested Action: Flag for review and monitor closely.

Enter text: I liked your post — g

In [ ]:
# -------------------------------------------
# 1. Import Libraries
# -------------------------------------------
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import CalibratedClassifierCV
import numpy as np
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# -------------------------------------------
# 2. Expanded Cyberbullying Dataset
# -------------------------------------------
data = {
    "text": [
        # ---------------- Not Cyberbullying ----------------
        "I liked your post, great job!",
        "You are very talented and kind.",
        "This is a good idea, thanks for sharing.",
        "The meeting was productive and smooth.",
        "You are doing really well, keep going!",
        "Amazing presentation today!",
        "Thanks for your help, much appreciated.",
        "Your project looks really solid.",
        "Great teamwork from everyone.",
        "I love your creativity in this project.",

        # ---------------- Gender Cyberbullying ----------------
        "Women can never understand technology.",
        "He is too weak to play sports, boys should be stronger.",
        "Girls shouldn't talk in serious discussions.",
        "Men are stupid and only cause problems.",
        "She is a useless woman who knows nothing.",
        "Boys like you shouldn’t even try coding.",
        "Girls are always bad at math.",
        "Men can’t handle multitasking like women.",
        "Women are too emotional to lead projects.",
        "Boys should stop whining and work harder.",

        # ---------------- Religion Cyberbullying ----------------
        "People from that religion are dangerous.",
        "Your religion is stupid and outdated.",
        "They should ban that religion completely.",
        "People of your faith are disgusting.",
        "No one respects that religion at all.",
        "Your beliefs are worthless.",
        "That religion always causes problems.",
        "Stop following that faith, it’s nonsense.",
        "People of your religion are always lying.",
        "Your religion is a joke.",

        # ---------------- Age Cyberbullying ----------------
        "You are too old to understand modern things.",
        "Old people like you should stop using social media.",
        "You are too young to have an opinion.",
        "Nobody wants someone your age on the team.",
        "You act like a child, grow up.",
        "Young people like you are too inexperienced.",
        "You’re too old to learn anything new.",
        "Children like you don’t understand responsibility.",
        "Old people are always slow and weak.",
        "You’re too young to be taken seriously.",

        # ---------------- Ethnicity Cyberbullying ----------------
        "People of that race are lazy.",
        "Your ethnicity makes you inferior.",
        "They should send people of your race back.",
        "That community is useless and uneducated.",
        "Your race is disgusting and dumb.",
        "People from that ethnicity are always troublemakers.",
        "Your community has no sense of discipline.",
        "They don’t belong in our team because of race.",
        "Your ethnicity is a problem for everyone.",
        "That race always causes conflicts.",

        # ---------------- Other Cyberbullying ----------------
        "I will find you and hurt you.",
        "You should disappear from the internet.",
        "Everyone hates you, just quit already.",
        "I will make you regret ever posting that.",
        "You are the most annoying person alive.",
        "Stop talking, nobody wants to hear you.",
        "You’re worthless and stupid.",
        "I hope you fail in everything.",
        "No one likes you in this group.",
        "Your existence is annoying.",

    ],

    "label": [
        # not_cyberbullying
        *["not_cyberbullying"]*10,
        # gender
        *["gender"]*10,
        # religion
        *["religion"]*10,
        # age
        *["age"]*10,
        # ethnicity
        *["ethnicity"]*10,
        # other_cyberbullying
        *["other_cyberbullying"]*10
    ]
}

df = pd.DataFrame(data)

# -------------------------------------------
# 3. Preprocessing
# -------------------------------------------
stop_words = set(stopwords.words("english"))
ps = PorterStemmer()

def preprocess(text):
    words = text.lower().split()
    words = [ps.stem(w) for w in words if w not in stop_words]
    return " ".join(words)

df["processed_text"] = df["text"].apply(preprocess)

# -------------------------------------------
# 4. Feature Extraction
# -------------------------------------------
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["processed_text"])

# Encode Labels
le = LabelEncoder()
y = le.fit_transform(df["label"])

# -------------------------------------------
# 5. Train SVM with Probability Calibration
# -------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

svc = LinearSVC()
model = CalibratedClassifierCV(svc)  # Gives probability output
model.fit(X_train, y_train)

print("✅ Model Training Completed!")
print(f"🔥 Training Accuracy: {model.score(X_train, y_train)*100:.2f}%")
print(f"🔥 Testing Accuracy: {model.score(X_test, y_test)*100:.2f}%")

# -------------------------------------------
# 6. Real-Time Prediction System
# -------------------------------------------
def get_severity(prob):
    if prob > 0.75:
        return "HIGH", "⚠️ Immediate action needed!"
    elif prob > 0.45:
        return "MEDIUM", "🚨 Flag for review."
    else:
        return "LOW", "ℹ️ Monitor occasionally."

print("\n🔵 Real-time Cyberbullying Detector: type 'exit' to stop.")

while True:
    text = input("\nEnter text: ")

    if text.lower() == "exit":
        print("Program Stopped.")
        break

    processed = preprocess(text)
    vector = vectorizer.transform([processed])

    pred_proba = model.predict_proba(vector)[0]   # probability array
    pred_index = np.argmax(pred_proba)
    label = le.inverse_transform([pred_index])[0]
    confidence = pred_proba[pred_index]

    severity, advice = get_severity(confidence)

    print(f"\nPredicted Category: {label}")
    print(f"Confidence: {confidence:.2f}  -> Severity: {severity}")
    print(f"Suggested Action: {advice}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


✅ Model Training Completed!
🔥 Training Accuracy: 100.00%
🔥 Testing Accuracy: 50.00%

🔵 Real-time Cyberbullying Detector: type 'exit' to stop.

Enter text: You are too young to know anything about leadership

Predicted Category: age
Confidence: 0.47  -> Severity: MEDIUM
Suggested Action: 🚨 Flag for review.

Enter text: Girls can’t code properly, it’s just not their thing.

Predicted Category: gender
Confidence: 0.44  -> Severity: LOW
Suggested Action: ℹ️ Monitor occasionally.

Enter text: People of your faith have no sense of right and wrong.

Predicted Category: religion
Confidence: 0.55  -> Severity: MEDIUM
Suggested Action: 🚨 Flag for review.

Enter text: Your race makes you weak and unreliable

Predicted Category: ethnicity
Confidence: 0.62  -> Severity: MEDIUM
Suggested Action: 🚨 Flag for review.

Enter text: If you post that again, I will make sure you regret it.

Predicted Category: other_cyberbullying
Confidence: 0.49  -> Severity: MEDIUM
Suggested Action: 🚨 Flag for review.

En